# Low-Resolution Robust Shift-Center Vecchia Simulation

Grid simulation version of the Amarel shift-center robustness study.

Purpose: test whether lagged conditioning sets should reuse current-time neighbor locations, or instead choose fresh nearest neighbors around predicted upstream centers under true/predicted advection mismatch.


In [1]:
import os
import sys
import time
import io
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.fft
import matplotlib.pyplot as plt

AMAREL_SRC = "/home/jl2815/tco"
LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
_src = AMAREL_SRC if os.path.exists(AMAREL_SRC) else LOCAL_SRC
sys.path.insert(0, _src)

from GEMS_TCO import kernels_vecchia
from GEMS_TCO import kernels_vecchia_advec
from GEMS_TCO import orderings as _orderings

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# Low-resolution shift-center nearest-neighbor test: lat x4, lon x2.
DELTA_LAT = 0.044 * 4
DELTA_LON = 0.063 * 2
T_STEPS = 8

print("DEVICE:", DEVICE)
print("SRC:", _src)
print("Grid resolution:", DELTA_LAT, DELTA_LON)



DEVICE: cpu
SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
Grid resolution: 0.176 0.126


## Settings And Robust Candidate Grid


In [2]:
LAT_RANGE = (-3.0, 2.0)
LON_RANGE = (121.0, 131.0)
MC_NUM_ITERS = 10  # raise to 30+ once the candidate grid looks right
SEED = 42

SMOOTH = 0.5
MM_COND_NUMBER = 100
NHEADS = 0
DAILY_STRIDE = 2

LBFGS_STEPS = 5
LBFGS_EVAL = 20
LBFGS_HIST = 10
INIT_NOISE = 0.7
SUPPRESS_FIT_PRINTS = True

RUN_ONE_SHOT_GODAMBE = False
HESSIAN_EPS = 1e-4
SCORE_EPS = 1e-5
H_RIDGE_SCALE = 1e-6
GODAMBE_J_METHOD = "block"
GODAMBE_BLOCK_LAT_WIDTH = 0.50
GODAMBE_BLOCK_LON_WIDTH = 0.50
GODAMBE_BLOCK_TIME_WIDTH = 2.0

BASE_TRUE_DICT = {
    "sigmasq": 10.0,
    "range_lat": 0.30,
    "range_lon": 0.40,
    "range_time": 2.0,
    "advec_lat": 0.08,
    "nugget": 2.5,
}

# Negative advec_lon means the upstream location at previous times is eastward.
# Keep -0.126 because it is exactly one lon cell at lon x2 low resolution.
TRUE_ADVEC_LON_LIST = [-0.10, -0.126, -0.16, -0.25]
DEFAULT_TRUE_ADVEC_LON = -0.126
TRUE_DICT = {**BASE_TRUE_DICT, "advec_lon": DEFAULT_TRUE_ADVEC_LON}  # compatibility for old helper snippets

# Positive values are eastward lag-1 center offsets used by the shifted-center candidates.
# On this lon x2 grid, 0.10/0.126/0.16 mostly resolve to one cell; 0.20/0.25 to about two cells.
PRED_LAG1_LON_OFFSETS = [0.10, 0.126, 0.16, 0.20, 0.25]
SHIFT_BUDGETS = [(16, 10), (14, 8), (18, 15)]
INCLUDE_FULL_LOCAL_BASELINE = True
INCLUDE_REDUCED_LOCAL_CONTROLS = True

BASE_A = 20
LOCAL_BEST_B, LOCAL_BEST_C = 18, 15
LOCAL_AGGR_B, LOCAL_AGGR_C = 16, 10

print(f"lon step = {DELTA_LON:.3f}")
print("true advec_lon values:", TRUE_ADVEC_LON_LIST)
print("predicted lag-1 eastward offsets:", PRED_LAG1_LON_OFFSETS)
print("predicted offsets in lon cells:", [round(x / DELTA_LON, 3) for x in PRED_LAG1_LON_OFFSETS])

def make_true_dict(advec_lon):
    return {**BASE_TRUE_DICT, "advec_lon": float(advec_lon)}

def offset_tag(x):
    sign = "p" if x >= 0 else "m"
    return sign + f"{abs(float(x)):.3f}".replace(".", "p")

def total_conditioning_std(a, b=0, c=0, uses_lag2=True):
    total = int(a) + int(1 + b)
    if uses_lag2:
        total += int(1 + c)
    return total

def total_conditioning_advec_center(a, b=0, c=0, uses_lag2=True):
    # same-location anchor + shifted-center anchor + shifted-center NN list at each lag.
    total = int(a) + int(1 + 1 + b)
    if uses_lag2:
        total += int(1 + 1 + c)
    return total

def std_spec(name, a, b, c, group, allocation, daily_stride=DAILY_STRIDE, uses_lag2=True):
    return {
        "name": name,
        "group": group,
        "kernel": "std",
        "limit_A": int(a),
        "limit_B": int(b),
        "limit_C": int(c),
        "daily_stride": int(daily_stride),
        "uses_lag2": bool(uses_lag2),
        "pred_lag1_lon_offset": 0.0,
        "lag1_lon_shift": 0.0,
        "lag2_lon_shift": 0.0,
        "lag1_ratio_actual": float(b / a) if a else np.nan,
        "lag2_ratio_actual": float(c / a) if (a and uses_lag2) else 0.0,
        "total_conditioning": total_conditioning_std(a, b, c, uses_lag2=uses_lag2),
        "allocation": allocation,
        "description": f"t: {a}; t-1: same loc + {b} local around current location; t-2: same loc + {c} local around current location",
    }

def advec_center_spec(a, b, c, pred_lag1_offset, group="shift_center_robust", daily_stride=DAILY_STRIDE):
    tag = offset_tag(pred_lag1_offset)
    name = f"ShiftCenter_A{a:02d}_B{b:02d}_C{c:02d}_O{tag}"
    return {
        "name": name,
        "group": group,
        "kernel": "advec_center",
        "limit_A": int(a),
        "limit_B": int(b),
        "limit_C": int(c),
        "daily_stride": int(daily_stride),
        "uses_lag2": True,
        "pred_lag1_lon_offset": float(pred_lag1_offset),
        "advec_lon_offset": float(pred_lag1_offset),
        "lag1_lon_shift": float(pred_lag1_offset),
        "lag2_lon_shift": float(2.0 * pred_lag1_offset),
        "pred_lag1_cells": float(pred_lag1_offset / DELTA_LON),
        "pred_lag2_cells": float(2.0 * pred_lag1_offset / DELTA_LON),
        "lag1_ratio_actual": float(b / a) if a else np.nan,
        "lag2_ratio_actual": float(c / a) if a else np.nan,
        "total_conditioning": total_conditioning_advec_center(a, b, c, uses_lag2=True),
        "allocation": f"shift-center: B={b}, C={c}, lag1 offset={pred_lag1_offset:.3f}",
        "description": f"t: {a}; t-1: same loc + shifted center + {b} NN around east-shifted center; t-2: same loc + shifted center + {c} NN around 2x shifted center",
    }

def build_model_specs():
    specs = []
    if INCLUDE_FULL_LOCAL_BASELINE:
        specs.append(std_spec(
            "FullLocal_A20_B20_C20", BASE_A, 20, 20,
            "local_controls", "full local baseline: 20/20/20",
        ))
    specs.append(std_spec(
        "BaseLocal_A20_B18_C15", BASE_A, LOCAL_BEST_B, LOCAL_BEST_C,
        "local_controls", "best local-only reduced baseline: 20/18/15",
    ))
    if INCLUDE_REDUCED_LOCAL_CONTROLS:
        specs.append(std_spec(
            "StdLocal_A20_B16_C10", BASE_A, LOCAL_AGGR_B, LOCAL_AGGR_C,
            "local_controls", "aggressive local-only reduction control: 20/16/10",
        ))
    for b, c in SHIFT_BUDGETS:
        for offset in PRED_LAG1_LON_OFFSETS:
            specs.append(advec_center_spec(BASE_A, b, c, offset))
    return {f"{spec['group']}::{spec['name']}": spec for spec in specs}

MODEL_SPECS = build_model_specs()
spec_df = pd.DataFrame(MODEL_SPECS).T
print("True-advection scenarios:", len(TRUE_ADVEC_LON_LIST))
print("Number of model fits per scenario:", len(MODEL_SPECS))
print("Total fits per MC iteration:", len(TRUE_ADVEC_LON_LIST) * len(MODEL_SPECS))
display(spec_df[[
    "group", "kernel", "limit_A", "limit_B", "limit_C", "pred_lag1_lon_offset",
    "pred_lag1_cells", "lag1_lon_shift", "lag2_lon_shift", "daily_stride",
    "allocation", "lag1_ratio_actual", "lag2_ratio_actual", "total_conditioning", "description",
]])


lon step = 0.126
true advec_lon values: [-0.1, -0.126, -0.16, -0.25]
predicted lag-1 eastward offsets: [0.1, 0.126, 0.16, 0.2, 0.25]
predicted offsets in lon cells: [0.794, 1.0, 1.27, 1.587, 1.984]
True-advection scenarios: 4
Number of model fits per scenario: 18
Total fits per MC iteration: 72


,group,kernel,limit_A,limit_B,limit_C,pred_lag1_lon_offset,pred_lag1_cells,lag1_lon_shift,lag2_lon_shift,daily_stride,allocation,lag1_ratio_actual,lag2_ratio_actual,total_conditioning,description
local_controls::FullLocal_A20_B20_C20,local_controls,std,20,20,20,0.0,NaN,0.0,0.0,2,full local baseline: 20/20/20,1.0,1.0,62,t: 20; t-1: same loc + 20 local around current...
local_controls::BaseLocal_A20_B18_C15,local_controls,std,20,18,15,0.0,NaN,0.0,0.0,2,best local-only reduced baseline: 20/18/15,0.9,0.75,55,t: 20; t-1: same loc + 18 local around current...
local_controls::StdLocal_A20_B16_C10,local_controls,std,20,16,10,0.0,NaN,0.0,0.0,2,aggressive local-only reduction control: 20/16/10,0.8,0.5,48,t: 20; t-1: same loc + 16 local around current...
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p100,shift_center_robust,advec_center,20,16,10,0.1,0.793651,0.1,0.2,2,"shift-center: B=16, C=10, lag1 offset=0.100",0.8,0.5,50,t: 20; t-1: same loc + shifted center + 16 NN ...
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p126,shift_center_robust,advec_center,20,16,10,0.126,1.0,0.126,0.252,2,"shift-center: B=16, C=10, lag1 offset=0.126",0.8,0.5,50,t: 20; t-1: same loc + shifted center + 16 NN ...
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p160,shift_center_robust,advec_center,20,16,10,0.16,1.269841,0.16,0.32,2,"shift-center: B=16, C=10, lag1 offset=0.160",0.8,0.5,50,t: 20; t-1: same loc + shifted center + 16 NN ...
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p200,shift_center_robust,advec_center,20,16,10,0.2,1.587302,0.2,0.4,2,"shift-center: B=16, C=10, lag1 offset=0.200",0.8,0.5,50,t: 20; t-1: same loc + shifted center + 16 NN ...
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p250,shift_center_robust,advec_center,20,16,10,0.25,1.984127,0.25,0.5,2,"shift-center: B=16, C=10, lag1 offset=0.250",0.8,0.5,50,t: 20; t-1: same loc + shifted center + 16 NN ...
shift_center_robust::ShiftCenter_A20_B14_C08_Op0p100,shift_center_robust,advec_center,20,14,8,0.1,0.793651,0.1,0.2,2,"shift-center: B=14, C=8, lag1 offset=0.100",0.7,0.4,46,t: 20; t-1: same loc + shifted center + 14 NN ...
shift_center_robust::ShiftCenter_A20_B14_C08_Op0p126,shift_center_robust,advec_center,20,14,8,0.126,1.0,0.126,0.252,2,"shift-center: B=14, C=8, lag1 offset=0.126",0.7,0.4,46,t: 20; t-1: same loc + shifted center + 14 NN ...


## Boundary-Aware Shift-Center Kernel


In [3]:
class fit_vecchia_lbfgs_shift_center_nn(kernels_vecchia_advec.fit_vecchia_lbfgs_advec):
    """Shift-center NN variant with same-location fallback outside the observed lon range."""

    def _build_advec_lookup(self, n_points):
        from sklearn.neighbors import BallTree

        if self.spatial_coords is not None:
            coords_np = np.asarray(self.spatial_coords[:n_points], dtype=np.float64)
        else:
            all_data = [torch.from_numpy(d) if isinstance(d, np.ndarray) else d
                        for d in self.input_map.values()]
            coords_raw = all_data[0][:n_points, :2].cpu().numpy().astype(np.float64)
            nan_mask = np.isnan(coords_raw).any(axis=1)
            coords_np = coords_raw.copy()
            coords_np[nan_mask] = np.array([0.0, 1000.0])

        tree = BallTree(np.radians(coords_np), metric="haversine")
        lats = coords_np[:, 0]
        lons = coords_np[:, 1]
        valid = ~np.isnan(coords_np).any(axis=1)
        lon_min = float(np.nanmin(lons[valid]))
        lon_max = float(np.nanmax(lons[valid]))

        def query_shift(mult):
            shifted_lons = lons + mult * self.advec_lon_offset
            outside = (~valid) | (shifted_lons < lon_min) | (shifted_lons > lon_max)
            q = np.column_stack([np.radians(lats), np.radians(shifted_lons)])
            _, idx = tree.query(q, k=1)
            out = idx.flatten().astype(np.int64)
            # If the true upstream point leaves the observed grid, use the same location.
            out[outside] = np.arange(n_points, dtype=np.int64)[outside]
            return out

        return query_shift(1.0), query_shift(2.0)


## Simulation Helpers


In [4]:
P_LABELS = ["sigmasq", "range_lat", "range_lon", "range_time", "advec_lat", "advec_lon", "nugget"]
P_COLS = ["sigmasq_est", "range_lat_est", "range_lon_est", "range_t_est", "advec_lat_est", "advec_lon_est", "nugget_est"]
SPATIAL_KEYS = ["sigmasq", "range_lat", "range_lon"]
ADVECTION_KEYS = ["advec_lat", "advec_lon"]

def transform_log_phi_to_physical(p):
    phi1, phi2, phi3, phi4 = (torch.exp(p[i]) for i in range(4))
    rlon = 1.0 / phi2
    return torch.stack([
        phi1 / phi2,
        rlon / torch.sqrt(phi3),
        rlon,
        rlon / torch.sqrt(phi4),
        p[4],
        p[5],
        torch.exp(p[6]),
    ])

def get_covariance_on_grid(lx, ly, lt, params):
    params = torch.clamp(params, min=-15.0, max=15.0)
    phi1, phi2, phi3, phi4 = (torch.exp(params[i]) for i in range(4))
    u_lat = lx - params[4] * lt
    u_lon = ly - params[5] * lt
    dist = torch.sqrt(u_lat.pow(2) * phi3 + u_lon.pow(2) + lt.pow(2) * phi4 + 1e-8)
    return (phi1 / phi2) * torch.exp(-dist * phi2)

def build_target_grid(lat_range, lon_range):
    lat0, lat1 = float(min(lat_range)), float(max(lat_range))
    lon0, lon1 = float(min(lon_range)), float(max(lon_range))
    n_lat = int(np.floor((lat1 - lat0) / DELTA_LAT + 1e-9)) + 1
    n_lon = int(np.floor((lon1 - lon0) / DELTA_LON + 1e-9)) + 1
    lats = lat0 + torch.arange(n_lat, device=DEVICE, dtype=DTYPE) * DELTA_LAT
    lons = lon0 + torch.arange(n_lon, device=DEVICE, dtype=DTYPE) * DELTA_LON
    lats = torch.round(lats * 10000) / 10000
    lons = torch.round(lons * 10000) / 10000
    g_lat, g_lon = torch.meshgrid(lats, lons, indexing="ij")
    grid_coords = torch.stack([g_lat.flatten(), g_lon.flatten()], dim=1)
    return lats, lons, grid_coords

def grid_resolution_report(lats, lons):
    lat_d = torch.diff(lats).detach().cpu().numpy()
    lon_d = torch.diff(lons).detach().cpu().numpy()
    return {
        "lat_min_step": float(lat_d.min()) if len(lat_d) else np.nan,
        "lat_max_step": float(lat_d.max()) if len(lat_d) else np.nan,
        "lon_min_step": float(lon_d.min()) if len(lon_d) else np.nan,
        "lon_max_step": float(lon_d.max()) if len(lon_d) else np.nan,
        "lat_first_last": (float(lats[0]), float(lats[-1])),
        "lon_first_last": (float(lons[0]), float(lons[-1])),
    }

def generate_field_values(n_lat, n_lon, t_steps, params):
    cpu = torch.device("cpu")
    f32 = torch.float32
    px, py, pt = 2 * n_lat, 2 * n_lon, 2 * t_steps
    lx = torch.arange(px, device=cpu, dtype=f32) * DELTA_LAT
    lx[px // 2:] -= px * DELTA_LAT
    ly = torch.arange(py, device=cpu, dtype=f32) * DELTA_LON
    ly[py // 2:] -= py * DELTA_LON
    lt = torch.arange(pt, device=cpu, dtype=f32)
    lt[pt // 2:] -= pt
    params_cpu = params.cpu().float()
    Lx, Ly, Lt = torch.meshgrid(lx, ly, lt, indexing="ij")
    C = get_covariance_on_grid(Lx, Ly, Lt, params_cpu)
    S = torch.fft.fftn(C)
    S.real = torch.clamp(S.real, min=0)
    noise = torch.fft.fftn(torch.randn(px, py, pt, device=cpu, dtype=f32))
    field = torch.fft.ifftn(torch.sqrt(S.real) * noise).real[:n_lat, :n_lon, :t_steps]
    return field.to(device=DEVICE, dtype=DTYPE)

def assemble_reg_map(field, grid_coords, true_params, t_offset=21.0):
    nugget_std = torch.sqrt(torch.exp(true_params[6]))
    n_grid = grid_coords.shape[0]
    field_flat = field.reshape(n_grid, field.shape[-1])
    reg_map = {}
    for t_idx in range(field.shape[-1]):
        key = f"t{t_idx}"
        t_val = float(t_offset + t_idx)
        dummy = torch.zeros(7, device=DEVICE, dtype=DTYPE)
        if t_idx > 0:
            dummy[t_idx - 1] = 1.0
        rows = torch.zeros((n_grid, 11), device=DEVICE, dtype=DTYPE)
        rows[:, :2] = grid_coords
        rows[:, 2] = field_flat[:, t_idx] + torch.randn(n_grid, device=DEVICE, dtype=DTYPE) * nugget_std
        rows[:, 3] = t_val
        rows[:, 4:] = dummy.unsqueeze(0).expand(n_grid, -1)
        reg_map[key] = rows.detach()
    return reg_map

def compute_grid_ordering(grid_coords, mm_cond_number):
    coords_np = grid_coords.detach().cpu().numpy()
    ord_mm = _orderings.maxmin_cpp(coords_np)
    nns = _orderings.find_nns_l2(locs=coords_np[ord_mm], max_nn=mm_cond_number)
    return ord_mm, nns

def true_to_log_params(true_dict):
    phi2 = 1.0 / true_dict["range_lon"]
    phi1 = true_dict["sigmasq"] * phi2
    phi3 = (true_dict["range_lon"] / true_dict["range_lat"]) ** 2
    phi4 = (true_dict["range_lon"] / true_dict["range_time"]) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4), true_dict["advec_lat"], true_dict["advec_lon"], np.log(true_dict["nugget"])]

def backmap_params(out_params):
    p = [x.item() if isinstance(x, torch.Tensor) else float(x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        "sigmasq": np.exp(p[0]) / phi2,
        "range_lat": rlon / phi3 ** 0.5,
        "range_lon": rlon,
        "range_time": rlon / phi4 ** 0.5,
        "advec_lat": p[4],
        "advec_lon": p[5],
        "nugget": np.exp(p[6]),
    }

def rmsre_for_keys(est, true_dict, keys, zero_thresh=0.01):
    vals = []
    for key in keys:
        tv = true_dict[key]
        ev = est[key]
        if abs(tv) >= zero_thresh:
            vals.append(((ev - tv) / abs(tv)) ** 2)
        else:
            vals.append(abs(ev - tv) ** 2)
    return float(np.sqrt(np.mean(vals)))

def relative_se_summary(se_by_key, denom_dict, keys, zero_thresh=0.01):
    vals = []
    for key in keys:
        denom = abs(denom_dict[key])
        if denom >= zero_thresh:
            vals.append((se_by_key[key] / denom) ** 2)
        else:
            vals.append(se_by_key[key] ** 2)
    return float(np.sqrt(np.mean(vals)))

def calculate_metrics(out_params, true_dict):
    est = backmap_params(out_params)
    return {
        "overall_rmsre": rmsre_for_keys(est, true_dict, P_LABELS),
        "spatial_rmsre": rmsre_for_keys(est, true_dict, SPATIAL_KEYS),
        "range_time_re": abs(est["range_time"] - true_dict["range_time"]) / abs(true_dict["range_time"]),
        "advec_rmsre": rmsre_for_keys(est, true_dict, ADVECTION_KEYS),
        "nugget_re": abs(est["nugget"] - true_dict["nugget"]) / abs(true_dict["nugget"]),
        "est": est,
    }

def make_random_init(rng, true_log, init_noise):
    noisy = list(true_log)
    for i in [0, 1, 2, 3, 6]:
        noisy[i] = true_log[i] + rng.uniform(-init_noise, init_noise)
    for i in [4, 5]:
        scale = max(abs(true_log[i]), 0.05)
        noisy[i] = true_log[i] + rng.uniform(-2 * scale, 2 * scale)
    return noisy


## Optional Godambe Helpers


In [5]:
def finite_diff_hessian(nll_fn, p, eps=HESSIAN_EPS):
    n = p.shape[0]
    H = torch.zeros(n, n, device=p.device, dtype=p.dtype)
    for i in range(n):
        p_p = p.detach().clone(); p_m = p.detach().clone()
        p_p[i] += eps; p_m[i] -= eps
        p_p.requires_grad_(True); p_m.requires_grad_(True)
        g_p = torch.autograd.grad(nll_fn(p_p), p_p)[0].detach()
        g_m = torch.autograd.grad(nll_fn(p_m), p_m)[0].detach()
        H[i] = (g_p - g_m) / (2.0 * eps)
    return (H + H.T) / 2.0

def vecchia_per_unit_target_coords(model):
    chunks = []
    if model.Heads_data is not None and model.Heads_data.shape[0] > 0:
        chunks.append(model.Heads_data[:, [0, 1, 3]].to(dtype=DTYPE))
    for X_b in [model.X_A, model.X_AB, model.X_ABC]:
        if X_b is not None and X_b.shape[0] > 0:
            chunks.append(X_b[:, -1, :].to(dtype=DTYPE))
    if not chunks:
        return torch.empty((0, 3), device=DEVICE, dtype=DTYPE)
    return torch.cat(chunks, dim=0)

def make_block_ids(target_coords):
    lat = target_coords[:, 0]
    lon = target_coords[:, 1]
    tim = target_coords[:, 2]
    lat_id = torch.floor((lat - lat.min()) / GODAMBE_BLOCK_LAT_WIDTH).to(torch.long)
    lon_id = torch.floor((lon - lon.min()) / GODAMBE_BLOCK_LON_WIDTH).to(torch.long)
    if GODAMBE_BLOCK_TIME_WIDTH is None or GODAMBE_BLOCK_TIME_WIDTH <= 0:
        time_id = torch.zeros_like(lat_id)
    else:
        time_id = torch.floor((tim - tim.min()) / GODAMBE_BLOCK_TIME_WIDTH).to(torch.long)
    n_lon = int(lon_id.max().item()) + 1 if lon_id.numel() else 1
    n_time = int(time_id.max().item()) + 1 if time_id.numel() else 1
    raw_id = (lat_id * n_lon + lon_id) * n_time + time_id
    _, block_id = torch.unique(raw_id, sorted=True, return_inverse=True)
    return block_id

def score_cov_per_unit_centered(score_mat):
    n_units = score_mat.shape[1]
    score_mean = score_mat.mean(dim=1)
    score_centered = score_mat - score_mean.unsqueeze(1)
    if n_units > 1:
        return score_centered @ score_centered.T / (n_units * (n_units - 1))
    return score_mat @ score_mat.T / max(n_units ** 2, 1)

def score_cov_block_cluster(score_mat, target_coords):
    n_units = score_mat.shape[1]
    scores = score_mat.T.contiguous()
    block_id = make_block_ids(target_coords)
    n_blocks = int(block_id.max().item()) + 1 if block_id.numel() else 0
    block_scores = torch.zeros((n_blocks, scores.shape[1]), device=DEVICE, dtype=DTYPE)
    block_scores.index_add_(0, block_id, scores)
    if n_blocks > 1:
        centered = block_scores - block_scores.mean(dim=0, keepdim=True)
        J = centered.T @ centered * (n_blocks / (n_blocks - 1)) / (n_units ** 2)
    else:
        J = block_scores.T @ block_scores / max(n_units ** 2, 1)
    return J, n_blocks

def compute_vecchia_godambe(model, raw_params, true_dict):
    p_hat = torch.tensor(raw_params[:7], device=DEVICE, dtype=DTYPE, requires_grad=True)

    def nll(p):
        return model.vecchia_batched_likelihood(p)

    H = finite_diff_hessian(nll, p_hat)
    eig = torch.linalg.eigvalsh(H).detach()
    h_abs_min = torch.clamp(torch.min(torch.abs(eig)), min=1e-12)
    h_cond = float((torch.max(torch.abs(eig)) / h_abs_min).detach().cpu())
    beta_hat = model.get_gls_beta(p_hat).detach()

    def per_unit_losses(p):
        return model.vecchia_per_unit_nll_terms(p, beta_hat)

    cols = []
    for k in range(p_hat.shape[0]):
        pp = p_hat.detach().clone(); pm = p_hat.detach().clone()
        pp[k] += SCORE_EPS; pm[k] -= SCORE_EPS
        with torch.no_grad():
            cols.append((per_unit_losses(pp) - per_unit_losses(pm)) / (2.0 * SCORE_EPS))
    score_mat = torch.stack(cols)
    n_units = score_mat.shape[1]
    target_coords = vecchia_per_unit_target_coords(model)
    if target_coords.shape[0] != n_units:
        raise RuntimeError(f"target/score mismatch: targets={target_coords.shape[0]}, scores={n_units}")

    score_mean = score_mat.mean(dim=1)
    p_grad = p_hat.detach().clone().requires_grad_(True)
    profile_grad = torch.autograd.grad(nll(p_grad), p_grad)[0].detach()
    score_grad_diff = profile_grad - score_mean

    J_uncentered = score_mat @ score_mat.T / (n_units ** 2)
    J_centered = score_cov_per_unit_centered(score_mat)
    J_block, n_blocks = score_cov_block_cluster(score_mat, target_coords)
    if GODAMBE_J_METHOD == "block":
        J_main = J_block
    elif GODAMBE_J_METHOD == "per_unit_centered":
        J_main = J_centered
    elif GODAMBE_J_METHOD == "per_unit_uncentered":
        J_main = J_uncentered
    else:
        raise ValueError(f"Unknown GODAMBE_J_METHOD={GODAMBE_J_METHOD!r}")

    eye = torch.eye(H.shape[0], device=DEVICE, dtype=DTYPE)
    h_scale = torch.clamp(torch.mean(torch.abs(torch.diag(H))), min=1.0)
    H_inv = torch.linalg.pinv(H + eye * h_scale * H_RIDGE_SCALE)
    Jac = torch.autograd.functional.jacobian(transform_log_phi_to_physical, p_hat)

    def summarize_J(J):
        G_raw = H_inv @ J @ H_inv
        G_phys = Jac @ G_raw @ Jac.T
        se = torch.sqrt(torch.clamp(torch.diag(G_phys), min=0.0)).detach().cpu().numpy()
        se_by_key = dict(zip(P_LABELS, [float(x) for x in se]))
        return se_by_key, {
            "spatial": relative_se_summary(se_by_key, true_dict, SPATIAL_KEYS),
            "overall": relative_se_summary(se_by_key, true_dict, P_LABELS),
            "advec": relative_se_summary(se_by_key, true_dict, ADVECTION_KEYS),
            "nugget": se_by_key["nugget"] / abs(true_dict["nugget"]),
        }

    se_main, rel_main = summarize_J(J_main)
    se_block, rel_block = summarize_J(J_block)
    se_centered, rel_centered = summarize_J(J_centered)
    se_uncentered, rel_uncentered = summarize_J(J_uncentered)
    return {
        "gim_j_method": GODAMBE_J_METHOD,
        "gim_n_units": int(n_units),
        "gim_n_blocks": int(n_blocks),
        "gim_h_cond_abs": h_cond,
        "gim_score_mean_max_abs": float(torch.max(torch.abs(score_mean)).detach().cpu()),
        "gim_profile_grad_max_abs": float(torch.max(torch.abs(profile_grad)).detach().cpu()),
        "gim_score_profile_diff_max_abs": float(torch.max(torch.abs(score_grad_diff)).detach().cpu()),
        "gim_spatial_rel_se": rel_main["spatial"],
        "gim_overall_rel_se": rel_main["overall"],
        "gim_advec_rel_se": rel_main["advec"],
        "gim_nugget_rel_se": rel_main["nugget"],
        "gim_spatial_rel_se_block": rel_block["spatial"],
        "gim_spatial_rel_se_perunit_centered": rel_centered["spatial"],
        "gim_spatial_rel_se_uncentered": rel_uncentered["spatial"],
        **{f"gim_se_{k}": v for k, v in se_main.items()},
    }


## Fit And Robust Monte Carlo


In [6]:
def fit_vecchia_spec(model_key, spec, reg_map_ord, nns_grid, ordered_grid_coords_np, initial_vals, true_dict, compute_godambe=False):
    params = [torch.tensor([val], device=DEVICE, dtype=DTYPE, requires_grad=True) for val in initial_vals]
    daily_stride = int(spec.get("daily_stride", DAILY_STRIDE))
    if spec["kernel"] == "std":
        model = kernels_vecchia.fit_vecchia_lbfgs(
            smooth=SMOOTH,
            input_map=reg_map_ord,
            nns_map=nns_grid,
            mm_cond_number=MM_COND_NUMBER,
            nheads=NHEADS,
            limit_A=spec["limit_A"],
            limit_B=spec["limit_B"],
            limit_C=spec["limit_C"],
            daily_stride=daily_stride,
        )
    elif spec["kernel"] == "advec_center":
        model = fit_vecchia_lbfgs_shift_center_nn(
            smooth=SMOOTH,
            input_map=reg_map_ord,
            nns_map=nns_grid,
            mm_cond_number=MM_COND_NUMBER,
            nheads=NHEADS,
            limit_A=spec["limit_A"],
            limit_B=spec["limit_B"],
            limit_C=spec["limit_C"],
            daily_stride=daily_stride,
            spatial_coords=ordered_grid_coords_np,
            advec_lon_offset=spec.get("pred_lag1_lon_offset", abs(true_dict["advec_lon"])),
        )
    else:
        raise ValueError(f"Unknown kernel: {spec['kernel']}")

    t0 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            model.precompute_conditioning_sets()
    else:
        model.precompute_conditioning_sets()
    precompute_s = time.time() - t0

    optimizer = model.set_optimizer(params, lr=1.0, max_iter=LBFGS_EVAL, history_size=LBFGS_HIST)
    t1 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            out, n_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    else:
        out, n_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t1

    metrics = calculate_metrics(out, true_dict)
    godambe = {}
    gim_s = 0.0
    if compute_godambe:
        t2 = time.time()
        godambe = compute_vecchia_godambe(model, [float(x) for x in out[:7]], true_dict)
        gim_s = time.time() - t2
    return out, float(out[-1]), int(n_iter), precompute_s, fit_s, gim_s, metrics, godambe

def run_experiment(num_iters=MC_NUM_ITERS, seed=SEED, true_advec_lon_list=None, compute_godambe=False, save_csv=True, csv_name=None):
    if true_advec_lon_list is None:
        true_advec_lon_list = TRUE_ADVEC_LON_LIST
    true_advec_lon_list = [float(x) for x in true_advec_lon_list]

    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    lats_grid, lons_grid, grid_coords = build_target_grid(LAT_RANGE, LON_RANGE)
    n_lat, n_lon = len(lats_grid), len(lons_grid)
    print(f"Grid: {n_lat} x {n_lon} x {T_STEPS} = {n_lat * n_lon * T_STEPS:,} rows")
    print("Actual resolution:", grid_resolution_report(lats_grid, lons_grid))
    print("True advec_lon scenarios:", true_advec_lon_list)
    print("Model specs:")
    display(pd.DataFrame(MODEL_SPECS).T[[
        "group", "kernel", "limit_A", "limit_B", "limit_C", "pred_lag1_lon_offset",
        "daily_stride", "uses_lag2", "allocation", "total_conditioning", "description",
    ]])
    ord_grid, nns_grid = compute_grid_ordering(grid_coords, MM_COND_NUMBER)
    ordered_grid_coords_np = grid_coords[ord_grid].detach().cpu().numpy()
    print("Ordering done")

    records = []
    for it in range(num_iters):
        print(f"\n=== Iteration {it + 1}/{num_iters} ===")
        for true_advec_lon in true_advec_lon_list:
            true_dict = make_true_dict(true_advec_lon)
            true_log = true_to_log_params(true_dict)
            true_params = torch.tensor(true_log, device=DEVICE, dtype=DTYPE)
            initial_vals = make_random_init(rng, true_log, INIT_NOISE)
            true_lag1_shift = abs(true_advec_lon)
            print(
                f"\n--- true advec_lon={true_advec_lon:.3f}; "
                f"upstream lag1 east shift={true_lag1_shift:.3f} ({true_lag1_shift / DELTA_LON:.2f} cells) ---"
            )

            field = generate_field_values(n_lat, n_lon, T_STEPS, true_params)
            reg_map = assemble_reg_map(field, grid_coords, true_params)
            del field
            reg_map_ord = {k: v[ord_grid] for k, v in reg_map.items()}

            for model_key, spec in MODEL_SPECS.items():
                try:
                    print(f"{model_key}: fitting", end="")
                    out, loss, n_iter, pre_s, fit_s, gim_s, metrics, godambe = fit_vecchia_spec(
                        model_key,
                        spec,
                        reg_map_ord,
                        nns_grid,
                        ordered_grid_coords_np,
                        initial_vals,
                        true_dict,
                        compute_godambe=compute_godambe,
                    )
                    est = metrics.pop("est")
                    pred_offset = float(spec.get("pred_lag1_lon_offset", 0.0))
                    row = {
                        "iter": it + 1,
                        "true_advec_lon": true_advec_lon,
                        "true_lag1_lon_shift": true_lag1_shift,
                        "true_lag1_cells": true_lag1_shift / DELTA_LON,
                        "model_key": model_key,
                        "model": spec["name"],
                        "group": spec["group"],
                        "kernel": spec["kernel"],
                        "allocation": spec["allocation"],
                        "limit_A": spec["limit_A"],
                        "limit_B": spec["limit_B"],
                        "limit_C": spec["limit_C"],
                        "daily_stride": spec.get("daily_stride", DAILY_STRIDE),
                        "uses_lag2": spec.get("uses_lag2", True),
                        "pred_lag1_lon_offset": pred_offset,
                        "pred_lag2_lon_offset": 2.0 * pred_offset,
                        "pred_lag1_cells": pred_offset / DELTA_LON if pred_offset else 0.0,
                        "pred_lag2_cells": 2.0 * pred_offset / DELTA_LON if pred_offset else 0.0,
                        "offset_abs_error": abs(pred_offset - true_lag1_shift) if pred_offset else np.nan,
                        "offset_cell_error": abs(pred_offset - true_lag1_shift) / DELTA_LON if pred_offset else np.nan,
                        "lag1_lon_shift": spec.get("lag1_lon_shift", 0.0),
                        "lag2_lon_shift": spec.get("lag2_lon_shift", 0.0),
                        "lag1_ratio_actual": spec["lag1_ratio_actual"],
                        "lag2_ratio_actual": spec["lag2_ratio_actual"],
                        "total_conditioning": spec["total_conditioning"],
                        "loss": round(loss, 6),
                        "overall_rmsre": round(metrics["overall_rmsre"], 6),
                        "spatial_rmsre": round(metrics["spatial_rmsre"], 6),
                        "range_time_re": round(metrics["range_time_re"], 6),
                        "advec_rmsre": round(metrics["advec_rmsre"], 6),
                        "nugget_re": round(metrics["nugget_re"], 6),
                        "precompute_s": round(pre_s, 3),
                        "fit_s": round(fit_s, 3),
                        "gim_s": round(gim_s, 3),
                        "total_s": round(pre_s + fit_s + gim_s, 3),
                        "fit_iter": n_iter,
                    }
                    row.update({f"true_{k}": float(v) for k, v in true_dict.items()})
                    row.update({
                        "sigmasq_est": round(est["sigmasq"], 6),
                        "range_lat_est": round(est["range_lat"], 6),
                        "range_lon_est": round(est["range_lon"], 6),
                        "range_t_est": round(est["range_time"], 6),
                        "advec_lat_est": round(est["advec_lat"], 6),
                        "advec_lon_est": round(est["advec_lon"], 6),
                        "nugget_est": round(est["nugget"], 6),
                    })
                    row.update(godambe)
                    records.append(row)
                    print(f" | loss={loss:.4f} spatial={metrics['spatial_rmsre']:.4f} overall={metrics['overall_rmsre']:.4f} advec={metrics['advec_rmsre']:.4f} time={pre_s + fit_s + gim_s:.1f}s")
                except Exception as exc:
                    print(f" | FAILED: {type(exc).__name__}: {exc}")
                    records.append({
                        "iter": it + 1,
                        "true_advec_lon": true_advec_lon,
                        "model_key": model_key,
                        "model": spec.get("name", model_key),
                        "group": spec.get("group"),
                        "kernel": spec.get("kernel"),
                        "error": repr(exc),
                    })
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    df = pd.DataFrame(records)
    if save_csv:
        out_dir = Path("log")
        out_dir.mkdir(exist_ok=True)
        if csv_name is None:
            csv_name = "sim_vecchia_lowres_x4_lonx2_shift_center_robust_results.csv"
        out_path = out_dir / csv_name
        df.to_csv(out_path, index=False)
        print("Saved:", out_path)
    return df


## Optional One-Shot Godambe


In [7]:
if RUN_ONE_SHOT_GODAMBE:
    df_godambe = run_experiment(
        num_iters=1,
        seed=SEED,
        true_advec_lon_list=[DEFAULT_TRUE_ADVEC_LON],
        compute_godambe=True,
        save_csv=True,
        csv_name="sim_vecchia_lowres_x4_lonx2_shift_center_robust_godambe_050126_results.csv",
    )
    display(df_godambe.sort_values(["true_advec_lon", "gim_spatial_rel_se"]))
else:
    df_godambe = pd.DataFrame()
    print("Skipping one-shot Godambe. Set RUN_ONE_SHOT_GODAMBE=True to run it for DEFAULT_TRUE_ADVEC_LON only.")


Skipping one-shot Godambe. Set RUN_ONE_SHOT_GODAMBE=True to run it for DEFAULT_TRUE_ADVEC_LON only.


## Robust Monte Carlo Sweep


In [8]:
df_mc = run_experiment(
    num_iters=MC_NUM_ITERS,
    seed=SEED,
    true_advec_lon_list=TRUE_ADVEC_LON_LIST,
    compute_godambe=False,
    save_csv=True,
    csv_name="sim_vecchia_lowres_x4_lonx2_shift_center_robust_mc_050126_results.csv",
)
df_mc.head()


Grid: 29 x 80 x 8 = 18,560 rows
Actual resolution: {'lat_min_step': 0.1759999999999997, 'lat_max_step': 0.17600000000000016, 'lon_min_step': 0.12599999999997635, 'lon_max_step': 0.12600000000000477, 'lat_first_last': (-3.0, 1.928), 'lon_first_last': (121.0, 130.954)}
True advec_lon scenarios: [-0.1, -0.126, -0.16, -0.25]
Model specs:


,group,kernel,limit_A,limit_B,limit_C,pred_lag1_lon_offset,daily_stride,uses_lag2,allocation,total_conditioning,description
local_controls::FullLocal_A20_B20_C20,local_controls,std,20,20,20,0.0,2,True,full local baseline: 20/20/20,62,t: 20; t-1: same loc + 20 local around current...
local_controls::BaseLocal_A20_B18_C15,local_controls,std,20,18,15,0.0,2,True,best local-only reduced baseline: 20/18/15,55,t: 20; t-1: same loc + 18 local around current...
local_controls::StdLocal_A20_B16_C10,local_controls,std,20,16,10,0.0,2,True,aggressive local-only reduction control: 20/16/10,48,t: 20; t-1: same loc + 16 local around current...
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p100,shift_center_robust,advec_center,20,16,10,0.1,2,True,"shift-center: B=16, C=10, lag1 offset=0.100",50,t: 20; t-1: same loc + shifted center + 16 NN ...
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p126,shift_center_robust,advec_center,20,16,10,0.126,2,True,"shift-center: B=16, C=10, lag1 offset=0.126",50,t: 20; t-1: same loc + shifted center + 16 NN ...
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p160,shift_center_robust,advec_center,20,16,10,0.16,2,True,"shift-center: B=16, C=10, lag1 offset=0.160",50,t: 20; t-1: same loc + shifted center + 16 NN ...
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p200,shift_center_robust,advec_center,20,16,10,0.2,2,True,"shift-center: B=16, C=10, lag1 offset=0.200",50,t: 20; t-1: same loc + shifted center + 16 NN ...
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p250,shift_center_robust,advec_center,20,16,10,0.25,2,True,"shift-center: B=16, C=10, lag1 offset=0.250",50,t: 20; t-1: same loc + shifted center + 16 NN ...
shift_center_robust::ShiftCenter_A20_B14_C08_Op0p100,shift_center_robust,advec_center,20,14,8,0.1,2,True,"shift-center: B=14, C=8, lag1 offset=0.100",46,t: 20; t-1: same loc + shifted center + 14 NN ...
shift_center_robust::ShiftCenter_A20_B14_C08_Op0p126,shift_center_robust,advec_center,20,14,8,0.126,2,True,"shift-center: B=14, C=8, lag1 offset=0.126",46,t: 20; t-1: same loc + shifted center + 14 NN ...


Ordering done

=== Iteration 1/10 ===

--- true advec_lon=-0.100; upstream lag1 east shift=0.100 (0.79 cells) ---
local_controls::FullLocal_A20_B20_C20: fitting | loss=1.5339 spatial=0.0270 overall=0.1479 advec=0.2657 time=45.1s
local_controls::BaseLocal_A20_B18_C15: fitting | loss=1.5487 spatial=0.0410 overall=0.1254 advec=0.2249 time=40.2s
local_controls::StdLocal_A20_B16_C10: fitting | loss=1.5428 spatial=0.0346 overall=0.1104 advec=0.1941 time=33.1s
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p100: fitting | loss=1.4552 spatial=0.0247 overall=0.0304 advec=0.0454 time=28.2s
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p126: fitting | loss=1.4552 spatial=0.0247 overall=0.0304 advec=0.0454 time=28.1s
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p160: fitting | loss=1.4645 spatial=0.0107 overall=0.0555 advec=0.0969 time=26.1s
shift_center_robust::ShiftCenter_A20_B16_C10_Op0p200: fitting | loss=1.4645 spatial=0.0194 overall=0.0446 advec=0.0711 time=37.8s
shift_center_robust::S

KeyboardInterrupt: 

In [ ]:
def p90_p10(x):
    return np.percentile(x, 90) - np.percentile(x, 10)

def add_relative_error_columns(df):
    out = df.copy()
    pair_map = {
        "sigmasq": ("sigmasq_est", "true_sigmasq"),
        "range_lat": ("range_lat_est", "true_range_lat"),
        "range_lon": ("range_lon_est", "true_range_lon"),
        "range_time": ("range_t_est", "true_range_time"),
        "advec_lat": ("advec_lat_est", "true_advec_lat"),
        "advec_lon": ("advec_lon_est", "true_advec_lon"),
        "nugget": ("nugget_est", "true_nugget"),
    }
    for par, (est_col, true_col) in pair_map.items():
        denom = out[true_col].abs().where(out[true_col].abs() > 1e-12, 1.0)
        out[f"{par}_re"] = (out[est_col] - out[true_col]).abs() / denom
    out["pred_offset_label"] = out["pred_lag1_lon_offset"].map(lambda x: "local" if pd.isna(x) or x == 0 else f"{x:.3f}")
    return out

def summarize_mc(df):
    df = add_relative_error_columns(df)
    metric_cols = [
        "loss", "overall_rmsre", "spatial_rmsre", "range_lat_re", "range_lon_re",
        "range_time_re", "advec_rmsre", "advec_lat_re", "advec_lon_re", "nugget_re", "total_s",
    ]
    rows = []
    ok = df.dropna(subset=["loss"])
    for (true_advec_lon, group, model), sub in ok.groupby(["true_advec_lon", "group", "model"], sort=False):
        row = {
            "true_advec_lon": float(true_advec_lon),
            "true_lag1_lon_shift": float(sub["true_lag1_lon_shift"].iloc[0]),
            "group": group,
            "model": model,
            "kernel": sub["kernel"].iloc[0],
            "allocation": sub["allocation"].iloc[0],
            "limit_A": int(sub["limit_A"].iloc[0]),
            "limit_B": int(sub["limit_B"].iloc[0]),
            "limit_C": int(sub["limit_C"].iloc[0]),
            "pred_lag1_lon_offset": float(sub["pred_lag1_lon_offset"].iloc[0]),
            "offset_abs_error": float(sub["offset_abs_error"].dropna().iloc[0]) if sub["offset_abs_error"].notna().any() else np.nan,
            "offset_cell_error": float(sub["offset_cell_error"].dropna().iloc[0]) if sub["offset_cell_error"].notna().any() else np.nan,
            "lag1_lon_shift": float(sub.get("lag1_lon_shift", pd.Series([0.0])).iloc[0]),
            "lag2_lon_shift": float(sub.get("lag2_lon_shift", pd.Series([0.0])).iloc[0]),
            "total_conditioning": int(sub["total_conditioning"].iloc[0]),
            "n": len(sub),
        }
        for col in metric_cols:
            vals = sub[col].dropna().to_numpy()
            row[f"{col}_mean"] = np.mean(vals)
            row[f"{col}_median"] = np.median(vals)
            row[f"{col}_p90_p10"] = p90_p10(vals)
        rows.append(row)
    return pd.DataFrame(rows)

def parameter_re_summary(df):
    df = add_relative_error_columns(df)
    cols = {
        "sigmasq": "sigmasq_re",
        "range_lat": "range_lat_re",
        "range_lon": "range_lon_re",
        "range_time": "range_time_re",
        "advec_lat": "advec_lat_re",
        "advec_lon": "advec_lon_re",
        "nugget": "nugget_re",
    }
    rows = []
    ok = df.dropna(subset=["loss"])
    for (true_advec_lon, group, model), sub in ok.groupby(["true_advec_lon", "group", "model"], sort=False):
        for par, col in cols.items():
            vals = sub[col].dropna()
            rows.append({
                "true_advec_lon": float(true_advec_lon),
                "group": group,
                "model": model,
                "kernel": sub["kernel"].iloc[0],
                "pred_lag1_lon_offset": float(sub["pred_lag1_lon_offset"].iloc[0]),
                "offset_abs_error": float(sub["offset_abs_error"].dropna().iloc[0]) if sub["offset_abs_error"].notna().any() else np.nan,
                "total_conditioning": int(sub["total_conditioning"].iloc[0]),
                "parameter": par,
                "mean_re": vals.mean(),
                "median_re": vals.median(),
                "p90_p10_re": p90_p10(vals.to_numpy()),
            })
    return pd.DataFrame(rows)

mc_summary = summarize_mc(df_mc)
param_summary = parameter_re_summary(df_mc)

rank_cols = [
    "true_advec_lon", "model", "kernel", "total_conditioning", "pred_lag1_lon_offset",
    "offset_abs_error", "n", "loss_mean", "overall_rmsre_mean", "spatial_rmsre_mean",
    "range_lon_re_mean", "range_time_re_mean", "advec_lat_re_mean", "advec_lon_re_mean",
    "nugget_re_mean", "total_s_mean",
]
print("Best models by true-advection scenario, sorted by overall RMSRE:")
display(
    mc_summary.sort_values(["true_advec_lon", "overall_rmsre_mean"])[rank_cols]
    .groupby("true_advec_lon", group_keys=False)
    .head(12)
)

print("Parameter recovery summary for temporal/advection-sensitive parameters:")
display(
    param_summary[param_summary["parameter"].isin(["advec_lat", "advec_lon", "range_time", "range_lon", "nugget"])]
    .sort_values(["true_advec_lon", "parameter", "mean_re"])
    .groupby(["true_advec_lon", "parameter"], group_keys=False)
    .head(8)
)


In [ ]:
for true_lon, sub in mc_summary.groupby("true_advec_lon", sort=True):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    shift = sub[sub["kernel"] == "advec_center"].copy()
    controls = sub[sub["kernel"] == "std"].copy()

    for (b, c), s in shift.groupby(["limit_B", "limit_C"], sort=True):
        s = s.sort_values("pred_lag1_lon_offset")
        label = f"shift B{int(b)}/C{int(c)}"
        axes[0].plot(s["pred_lag1_lon_offset"], s["overall_rmsre_mean"], marker="o", label=label)
        axes[1].plot(s["pred_lag1_lon_offset"], s["advec_rmsre_mean"], marker="o", label=label)
        axes[2].plot(s["pred_lag1_lon_offset"], s["range_time_re_mean"], marker="o", label=label)

    for _, r in controls.iterrows():
        axes[0].axhline(r["overall_rmsre_mean"], linestyle="--", alpha=0.45, label=r["model"])
        axes[1].axhline(r["advec_rmsre_mean"], linestyle="--", alpha=0.45, label=r["model"])
        axes[2].axhline(r["range_time_re_mean"], linestyle="--", alpha=0.45, label=r["model"])

    true_shift = abs(true_lon)
    for ax in axes:
        ax.axvline(true_shift, color="black", linewidth=1.2, alpha=0.7)
        ax.set_xlabel("predicted lag-1 eastward offset")
        ax.grid(alpha=0.2)

    axes[0].set_title(f"true advec_lon={true_lon:.3f}: overall RMSRE")
    axes[1].set_title("advection RMSRE")
    axes[2].set_title("range_time RE")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## Diagnostic Decision Rule


In [ ]:
print("Decision rule:")
print("1. BaseLocal_A20_B18_C15 is the best local-only reduced baseline from earlier tests.")
print("2. StdLocal_A20_B16_C10 tells us the cost of aggressive local-only reduction without shifted centers.")
print("3. ShiftCenter candidates test whether fresh lagged NN around predicted upstream centers can beat 20/18/15 while using fewer conditioning slots.")
print("4. Robustness matters: exact-offset wins are not enough; check whether nearby predicted offsets still beat or tie 20/18/15 when true advection changes.")
print("5. Because lon x2 low-res has DELTA_LON=0.126, 0.10/0.126/0.16 are mostly a one-cell test and 0.20/0.25 are mostly a two-cell test.")
display(spec_df[["limit_A", "limit_B", "limit_C", "kernel", "pred_lag1_lon_offset", "total_conditioning", "allocation"]])


## Interpretation

- If `ShiftCenter_A20_B16_C10` beats `BaseLocal_A20_B18_C15` near the correct offset, then reusing the current-time neighbor locations at lagged times is probably not necessary for this simulation.
- If `ShiftCenter_A20_B14_C08` also stays competitive, then the shifted-center idea may reduce conditioning budget beyond the previous local-only optimum.
- If performance collapses when the predicted offset is misspecified, the method is useful only with a reasonably good advection prior or an adaptive offset search.
- On this low-resolution lon x2 grid, offset differences below about half a lon cell may collapse to the same effective neighbor set, so use this notebook as a fast diagnostic before trusting the irregular/high-resolution Amarel run.
